# 00 — GPU / Environment Preflight

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Section 13 rule:** If `torch.cuda.is_available() == False` → STOP. Never silently fall back to CPU.
>
> **Section 14 rule:** ONNX `CUDAExecutionProvider` must be verified before any official GPU inference benchmark.

---

## Sections
1. Environment & System Info
2. GPU Hardware Check
3. PyTorch + CUDA Verification
4. Key Package Inventory
5. ONNX Runtime Provider Check
6. Disk Space
7. Preflight Summary

## 1. Environment & System Info

In [ ]:
import os, sys, platform, subprocess, datetime

print('=' * 60)
print('  SIH PS 26168 — GPU / Environment Preflight')
print(f'  Run at: {datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")}')
print('=' * 60)
print(f'Hostname     : {platform.node()}')
print(f'OS           : {platform.platform()}')
print(f'Python       : {sys.version}')
print(f'Working dir  : {os.getcwd()}')
print(f'Python bin   : {sys.executable}')

## 2. GPU Hardware Check

In [ ]:
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('nvidia-smi failed — GPU not available. STOP per Section 13.')
print(result.stdout)

# Compact GPU summary
r2 = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU summary:')
print(r2.stdout)

## 3. PyTorch + CUDA Verification

In [ ]:
import torch

print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError(
        'FATAL: CUDA is NOT available.\n'
        'Per PROJECT_RULES.md Section 7 and roadmap Section 13:\n'
        'Never silently fall back to CPU. Fix the CUDA environment before proceeding.'
    )

print(f'CUDA version     : {torch.version.cuda}')
print(f'cuDNN version    : {torch.backends.cudnn.version()}')
print(f'cuDNN enabled    : {torch.backends.cudnn.enabled}')
print(f'Number of GPUs   : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    vram_gb = p.total_memory / 1024**3
    print(f'  GPU {i}: {p.name} | VRAM: {vram_gb:.1f} GB | Compute: {p.major}.{p.minor}')

# Quick smoke test
t = torch.randn(128, 128, device='cuda')
result = (t @ t.T).mean().item()
print(f'\nGPU matmul smoke test: {result:.4f} ✓')
print('\nPyTorch CUDA: PASSED')

## 4. Key Package Inventory

In [ ]:
import importlib

packages = [
    ('numpy',         'numpy'),
    ('pandas',        'pandas'),
    ('scipy',         'scipy'),
    ('matplotlib',    'matplotlib'),
    ('seaborn',       'seaborn'),
    ('sklearn',       'scikit-learn'),
    ('torch',         'pytorch'),
    ('tqdm',          'tqdm'),
    ('yaml',          'pyyaml'),
    ('onnx',          'onnx'),
    ('onnxruntime',   'onnxruntime-gpu'),
    ('torch_geometric','torch_geometric (PyG)'),
    ('papermill',     'papermill'),
    ('pyproj',        'pyproj'),
    ('osmnx',         'osmnx'),
    ('networkx',      'networkx'),
]

missing = []
print(f'{"Package":<25} {"Version":<20} {"Status"}')
print('-' * 60)
for import_name, display_name in packages:
    try:
        m = importlib.import_module(import_name)
        ver = getattr(m, '__version__', 'unknown')
        print(f'{display_name:<25} {ver:<20} OK')
    except ImportError:
        print(f'{display_name:<25} {"---":<20} MISSING')
        missing.append(display_name)

if missing:
    print(f'\nMissing packages: {missing}')
    print('Install with: pip install', ' '.join(missing))
else:
    print('\nAll packages present.')

## 5. ONNX Runtime Provider Check

In [ ]:
try:
    import onnxruntime as ort
    providers = ort.get_available_providers()
    print(f'ONNX Runtime version    : {ort.__version__}')
    print(f'Available providers     : {providers}')
    if 'CUDAExecutionProvider' in providers:
        print('\nONNX GPU: CUDAExecutionProvider is AVAILABLE ✓')
    else:
        print('\nWARNING: CUDAExecutionProvider NOT in available providers.')
        print('Per PROJECT_RULES.md Section 7 and roadmap Section 14:')
        print('Official GPU inference benchmark cannot proceed without it.')
        print('Install: pip install onnxruntime-gpu')
except ImportError:
    print('onnxruntime not installed — install onnxruntime-gpu')

## 6. Disk Space

In [ ]:
import shutil

paths_to_check = ['.', '/home/zeus', '/tmp']
print(f'{"Path":<30} {"Total":>10} {"Used":>10} {"Free":>10}')
print('-' * 64)
for p in paths_to_check:
    if os.path.exists(p):
        total, used, free = shutil.disk_usage(p)
        print(f'{p:<30} {total/1e9:>9.1f}G {used/1e9:>9.1f}G {free/1e9:>9.1f}G')

## 7. Preflight Summary

In [ ]:
import json, datetime

summary = {
    'timestamp_utc': datetime.datetime.utcnow().isoformat(),
    'hostname': platform.node(),
    'python_version': sys.version,
    'pytorch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_version': torch.version.cuda if torch.cuda.is_available() else None,
    'cudnn_version': str(torch.backends.cudnn.version()) if torch.cuda.is_available() else None,
    'num_gpus': torch.cuda.device_count(),
    'gpus': [
        {
            'index': i,
            'name': torch.cuda.get_device_properties(i).name,
            'vram_gb': round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2)
        }
        for i in range(torch.cuda.device_count())
    ],
    'missing_packages': missing if 'missing' in dir() else [],
    'preflight_status': 'PASSED' if torch.cuda.is_available() else 'FAILED'
}

os.makedirs('artifacts', exist_ok=True)
with open('artifacts/gpu_preflight.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print('\nSaved: artifacts/gpu_preflight.json')
print('\n' + '=' * 60)
print(f'  PREFLIGHT: {summary["preflight_status"]}')
print('=' * 60)